# PSD Mean Baseline Analysis
This notebook executes the baseline experiments (Tests 1 through 5) utilizing exclusively the Power Spectrum Density (`psd_mean`) metric. It covers both the Semi-Urban and Forest datasets across four distinct classifiers (RF, XGB, LR, SGD) and multiple granularity/content configurations.

In [1]:
import os
import sys
import logging
from pathlib import Path
from typing import Tuple, Dict, Any, List

import pandas as pd
import numpy as np
import joblib

current_path = Path.cwd()
PROJECT_ROOT = None

for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Project root added to sys.path: {PROJECT_ROOT}")
else:
    print("❌ ERROR: Unable to find the project root.")

from rainfall_acoustic_classification.utils import get_standard_logger
from rainfall_acoustic_classification.feature_engineering import (
    build_single_selector, 
    SingleSelectorConfig,
    ExperimentConfig,
    ExperimentCreator,
    plot_scaled_distributions,
    plot_individual_boxplots
)
from rainfall_acoustic_classification.modeling import (
    ClassifierFactory, 
    ClassifierConfig,
    ModelOptimizer,
    TuningConfig,
    ModelEvaluator, 
    ValidationConfig,
    plot_confusion_matrix_grid,
    plot_multiclass_pr_curve,
    plot_experiment_performance_heatmap
)
from rainfall_acoustic_classification.visualization import VisualizationEngine

# Explicit Global Configuration
RANDOM_STATE = 42
N_JOBS = -1
TARGET_COLUMN = 'category' # Update if your label column has a different name
DATA_DIR = Path(f'{PROJECT_ROOT}/data/processed')
MODEL_DIR = Path(f'{PROJECT_ROOT}/models')
REPORTS_DIR = Path(f'{PROJECT_ROOT}/reports')

# Ensure output directory exists
MODEL_DIR.mkdir(parents=True, exist_ok=True)

logger = get_standard_logger("PSD_Baseline_Notebook")

viz_engine = VisualizationEngine()
logger.info("Visualization Engine instantiated. Core directories verified.")

Project root added to sys.path: /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026


/Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


16:16:15 - [PSD_Baseline_Notebook] - INFO - Visualization Engine instantiated. Core directories verified.


In [2]:
# ==============================================================================
# Cell 2: Data Ingestion
# ==============================================================================
def load_dataset_splits(
    dataset_name: str, 
    data_dir: Path
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Loads train, validation, and test splits from processed CSV files.

    Parameters
    ----------
    dataset_name : str
        The identifier for the dataset (e.g., 'semi_urban', 'forest').
    data_dir : pathlib.Path
        The base directory containing the processed data.

    Returns
    -------
    tuple of pandas.DataFrame
        A tuple containing (df_train, df_val, df_test).
    """
    train_path = data_dir / f"{dataset_name}" / f"{dataset_name}_train_metrics.csv"
    val_path = data_dir / f"{dataset_name}" / f"{dataset_name}_val_metrics.csv"
    test_path = data_dir / f"{dataset_name}" / f"{dataset_name}_test_metrics.csv"
    
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)
    
    logger.info(f"Loaded {dataset_name} | Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
    return df_train, df_val, df_test

def prepare_experiment_data(
    df: pd.DataFrame,
    content: str,
    granularity: int,
    target_col: str
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Applies the ExperimentCreator rules to extract and mutate the dataset.

    Parameters
    ----------
    df : pandas.DataFrame
        The raw dataset split.
    content : str
        The acoustic content filter ('dry/wet', 'wet', 'dry').
    granularity : int
        The target number of classes.
    target_col : str
        The name of the label column.

    Returns
    -------
    tuple
        A tuple containing the mutated Feature Matrix (X) and Target Vector (y).
    """
    exp_config = ExperimentConfig(
        content=content,
        granularity=granularity,
        label_col=target_col
    )

    X_raw, y_raw, meta = ExperimentCreator.extract_X_y_meta(df, config=exp_config)
    X_mut, y_mut, _ = ExperimentCreator.apply_experiment_rules(X_raw, y_raw, meta=meta, config=exp_config)
    
    return X_mut, y_mut

In [3]:
# ==============================================================================
# Cell 3: Atomic Functions - Visualization Orchestrator
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

def get_canonical_class_order(present_classes: List[str]) -> List[str]:
    """
    Enforces the physical intensity progression for acoustic rain categories.
    """
    MASTER_ORDER = [
        'No Rain', 'Dry', 
        'Wet', 'Light', 'Light/Moderate', 
        'Moderate', 'Heavy', 'Heavy/Violent', 'Violent'
    ]
    ordered = [c for c in MASTER_ORDER if c in present_classes]
    ordered += [c for c in present_classes if c not in MASTER_ORDER]
    return ordered

def generate_physical_boxplots(
    X: pd.DataFrame, 
    y: pd.Series, 
    feature_name: str, 
    class_order: List[str], 
    color_map: Dict[str, str], 
    save_path: Path
) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_df = X.copy()
    plot_df['Target'] = y.values
    
    sns.boxplot(
        data=plot_df, x='Target', y=feature_name, hue='Target',
        palette=color_map, order=class_order, hue_order=class_order,
        legend=False, ax=ax,
        flierprops=dict(marker='o', markersize=4, markerfacecolor='red', markeredgecolor='red', alpha=0.5)
    )
    
    ax.set_title(f"Physical Distribution: {feature_name}", fontsize=14)
    ax.set_xlabel("Rainfall Intensity", fontsize=12)
    ax.set_ylabel(f"{feature_name} (Raw Units)", fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    fig.savefig(str(save_path), dpi=300, format='pdf', bbox_inches='tight')
    plt.close(fig)

def generate_experiment_plots(
    metrics: Dict[str, Any],
    X_train_scaled: pd.DataFrame,
    y_train: pd.Series,
    dataset_name: str,
    clf_type: str,
    exp_id: int,
    feature_name: str,
    engine: VisualizationEngine,
    save_dir: Path
) -> None:
    exp_dir = save_dir / dataset_name / f"exp_{exp_id}_{clf_type}"
    exp_dir.mkdir(parents=True, exist_ok=True)
    prefix = f"{dataset_name}_E{exp_id}_{clf_type}"
    
    classes = get_canonical_class_order(list(metrics['classes']))
    
    global_palette = engine.get_rain_color_palette()
    experiment_palette = {c: global_palette.get(c, '#808080') for c in classes}
    
    try:
        plot_scaled_distributions(
            X=X_train_scaled, y=y_train, color_map=experiment_palette,
            class_order=classes, features_to_plot=[feature_name],
            title=f"Z-Score Distribution ({feature_name}) - E{exp_id}",
            save_path=str(exp_dir / f"{prefix}_scaled_dist.pdf")
        )
    except Exception as e:
        logger.warning(f"Could not generate Scaled Distributions plot: {e}")

    if 'confusion_matrix_raw' in metrics:
        plot_confusion_matrix_grid(
            cm_raw=metrics['confusion_matrix_raw'], cm_norm=metrics['confusion_matrix_normalized'],
            classes=classes, title=f"Confusion Matrix: {clf_type.upper()} - {dataset_name} (E{exp_id})",
            save_path=str(exp_dir / f"{prefix}_cm.pdf")
        )

    if metrics.get('y_proba') is not None and metrics.get('y_test_true') is not None:
        plot_multiclass_pr_curve(
            y_true=metrics['y_test_true'], y_proba=metrics['y_proba'],
            color_map=experiment_palette, class_order=classes,
            title=f"PR Curves: {clf_type.upper()} - {dataset_name} (E{exp_id})",
            save_path=str(exp_dir / f"{prefix}_pr_curve.pdf")
        )

def flatten_metrics_dict(d: dict, parent_key: str = '', sep: str = '_') -> dict:
    """Recursively flattens a nested dictionary, safely ignoring un-serializable objects like arrays."""
    items = []
    for k, v in d.items():
        # Ignora objetos massivos e matrizes que não cabem numa célula de CSV
        if isinstance(v, (np.ndarray, list, tuple, pd.Series, pd.DataFrame)):
            continue
        if k in ['classes', 'y_proba', 'y_test_true', 'confusion_matrix_raw', 'confusion_matrix_normalized']:
            continue
            
        clean_k = str(k).replace(' ', '_').replace('/', '_')
        new_key = f"{parent_key}{sep}{clean_k}" if parent_key else clean_k
        
        if isinstance(v, dict):
            items.extend(flatten_metrics_dict(v, new_key, sep=sep).items())
        elif isinstance(v, (int, float, str, bool)):
            items.append((new_key, v))
    return dict(items)

In [4]:
# ==============================================================================
# Cell 4: Decoupled Experiment Loop with Hyperparameter Tuning
# ==============================================================================
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import label_binarize

EXPERIMENT_MATRIX = [
    {'exp_id': 1, 'feature': 'psd_mean', 'content': 'dry/wet', 'granularity': 2},
    {'exp_id': 2, 'feature': 'psd_mean', 'content': 'dry/wet', 'granularity': 3},
    {'exp_id': 3, 'feature': 'psd_mean', 'content': 'dry/wet', 'granularity': 5},
    {'exp_id': 4, 'feature': 'psd_mean', 'content': 'wet', 'granularity': 2},
    {'exp_id': 5, 'feature': 'psd_mean', 'content': 'wet', 'granularity': 4},
]

DATASETS = ['IDSM', 'UECE']
CLASSIFIERS = ['rf', 'lr', 'sgd'] # 'xgb' is structurally prepared but suspended

HYPERPARAM_GRIDS = {
    'rf': {'n_estimators': [50, 100, 200], 'max_depth': [None, 3, 5], 'min_samples_leaf': [1, 5]},
    'lr': {'C': [0.1, 1.0, 10.0], 'class_weight': ['balanced']},
    'sgd': {'alpha': [0.0001, 0.001, 0.01], 'penalty': ['l2', 'elasticnet'], 'class_weight': ['balanced']},
    'xgb': {'n_estimators': [50, 100], 'max_depth': [3, 5], 'learning_rate': [0.01, 0.1]}
}

all_results: List[Dict[str, Any]] = []

for dataset in DATASETS:
    logger.info(f"========== INITIATING DATASET: {dataset.upper()} ==========")
    
    try:
        df_train, df_val, df_test = load_dataset_splits(dataset, DATA_DIR)
    except FileNotFoundError as e:
        logger.error(f"Data not found for {dataset}. Skipping. ({e})")
        continue
        
    for exp in EXPERIMENT_MATRIX:
        logger.info(f">>> Processing Experiment {exp['exp_id']} | {exp['content']} | {exp['granularity']}-class <<<")
        
        try:
            X_tr_raw, y_tr = prepare_experiment_data(df_train, exp['content'], exp['granularity'], TARGET_COLUMN)
            X_val_raw, y_val = prepare_experiment_data(df_val, exp['content'], exp['granularity'], TARGET_COLUMN)
            X_te_raw, y_te = prepare_experiment_data(df_test, exp['content'], exp['granularity'], TARGET_COLUMN)

            selector_config = SingleSelectorConfig(target_feature=exp['feature'])
            pipeline = build_single_selector(config=selector_config)
            
            X_tr_proc_np = pipeline.fit_transform(X_tr_raw, y_tr)
            X_val_proc_np = pipeline.transform(X_val_raw)
            X_te_proc_np = pipeline.transform(X_te_raw)

            X_tr_proc = pd.DataFrame(X_tr_proc_np, columns=[exp['feature']])
            X_val_proc = pd.DataFrame(X_val_proc_np, columns=[exp['feature']])
            X_te_proc = pd.DataFrame(X_te_proc_np, columns=[exp['feature']])

            # BOXPLOTS
            exp_data_dir = REPORTS_DIR / dataset / f"exp_{exp['exp_id']}_data"
            exp_data_dir.mkdir(parents=True, exist_ok=True)
            
            ordered_classes = get_canonical_class_order(list(y_tr.unique()))
            global_palette = viz_engine.get_rain_color_palette()
            exp_palette = {c: global_palette.get(c, '#808080') for c in ordered_classes}
            
            generate_physical_boxplots(
                X=X_tr_raw[[exp['feature']]], y=y_tr, feature_name=exp['feature'], 
                class_order=ordered_classes, color_map=exp_palette, 
                save_path=(exp_data_dir / f"{dataset}_E{exp['exp_id']}_physical_boxplot.pdf")
            )

            for clf in CLASSIFIERS:
                logger.info(f"--- Tuning & Training: {clf.upper()} ---")
                
                base_estimator = ClassifierFactory.build(
                    config=ClassifierConfig(model_type=clf, random_state=RANDOM_STATE, n_jobs=1)
                )
                
                tuning_config = TuningConfig(param_grid=HYPERPARAM_GRIDS[clf], scoring_metric='f1_macro', n_jobs=N_JOBS)
                
                champion_model = ModelOptimizer.optimize(
                    estimator=base_estimator, X_train=X_tr_proc, y_train=y_tr, 
                    X_val=X_val_proc, y_val=y_val, config=tuning_config
                )

                val_config = ValidationConfig(average_method='macro', return_report_dict=True)
                metrics = ModelEvaluator.evaluate(model=champion_model, X_test=X_te_proc, y_test=y_te, config=val_config)
                metrics['y_test_true'] = y_te

                # =========================================================================
                # NOVO: Injeção Explícita de PR-AUC Individual por Classe
                # =========================================================================
                classes_arr = list(metrics.get('classes', []))
                y_proba_arr = metrics.get('y_proba')
                
                if y_proba_arr is not None and len(classes_arr) > 1:
                    # Binarização necessária para o average_precision_score no estilo OVR
                    y_bin = label_binarize(y_te, classes=classes_arr)
                    # Tratamento do caso binário (label_binarize retorna array 1D)
                    if len(classes_arr) == 2:
                        y_bin = np.hstack((1 - y_bin, y_bin))
                        
                    for i, cls_name in enumerate(classes_arr):
                        safe_cls_name = str(cls_name).replace(' ', '_').replace('/', '_')
                        try:
                            ap_score = average_precision_score(y_bin[:, i], y_proba_arr[:, i])
                            # Salva a métrica na raiz do dicionário
                            metrics[f'PR_AUC_Class_{safe_cls_name}'] = ap_score
                        except Exception as e:
                            logger.warning(f"Could not calculate PR-AUC for class {cls_name}: {e}")
                # =========================================================================

                generate_experiment_plots(
                    metrics=metrics, X_train_scaled=X_tr_proc, y_train=y_tr,
                    dataset_name=dataset, clf_type=clf, exp_id=exp['exp_id'],
                    feature_name=exp['feature'], engine=viz_engine, save_dir=REPORTS_DIR
                )

                artifact_prefix = MODEL_DIR / f"{dataset}" / f"{dataset}_E{exp['exp_id']}_{clf}"
                joblib.dump(pipeline, f"{artifact_prefix}_pipeline.pkl")
                joblib.dump(champion_model, f"{artifact_prefix}_champion_model.pkl")

                # =========================================================================
                # NOVO: Flattening Amplo para Coletar Tudo
                # =========================================================================
                base_info = {
                    'Dataset': dataset, 'Exp_ID': exp['exp_id'], 'Algorithm': clf.upper(),
                    'Content': exp['content'], 'Granularity': f"{exp['granularity']}-Class"
                }
                
                # Extrai o dict do classification_report e o dicionário principal separadamente
                report_dict = metrics.pop('report_dict', {}) 
                
                flat_report = flatten_metrics_dict(report_dict)
                flat_metrics = flatten_metrics_dict(metrics) # Aqui o PR_AUC_Class será capturado
                
                # Combina tudo numa única linha horizontal do DataFrame
                final_row = {**base_info, **flat_report, **flat_metrics}
                all_results.append(final_row)

        except Exception as e:
            logger.error(f"Failure in Experiment {exp['exp_id']} for dataset {dataset}. Reason: {str(e)}")

# Save the consolidated tracking dataframe
df_all_results = pd.DataFrame(all_results)
df_all_results.to_csv(f'{PROJECT_ROOT}/models/baseline_experiments_full_metrics.csv', index=False)
display(df_all_results.head())
logger.info("Pipeline execution finished. Full metrics, including individual PR-AUCs, are saved.")

16:16:16 - [PSD_Baseline_Notebook] - INFO - ========== INITIATING DATASET: IDSM ==========


/var/folders/cn/k3rr2h_90qz9g2j3qxblfskc0000gn/T/ipykernel_705/3546309922.py:27: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(train_path)


16:16:17 - [PSD_Baseline_Notebook] - INFO - Loaded IDSM | Train: 28331 | Val: 2618 | Test: 2618
16:16:17 - [PSD_Baseline_Notebook] - INFO - >>> Processing Experiment 1 | dry/wet | 2-class <<<
16:16:17 - [ExperimentCreator] - INFO - Applying Smart Rules -> Content: 'dry/wet' | Granularity: 2
16:16:17 - [ExperimentCreator] - INFO - Applying Smart Rules -> Content: 'dry/wet' | Granularity: 2
16:16:17 - [ExperimentCreator] - INFO - Applying Smart Rules -> Content: 'dry/wet' | Granularity: 2
16:16:18 - [FeatureSelector] - INFO - Isolated 'psd_mean' | Fisher Score (F-Value): 3188.32
16:16:19 - [PSD_Baseline_Notebook] - INFO - --- Tuning & Training: RF ---
16:16:19 - [ClassifierFactory] - INFO - CPU Scaling: Requested 1 -> Allocated 1 cores.
16:16:19 - [ClassifierFactory] - INFO - Building registered model architecture: RF
16:16:19 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 18 candidates, totalling 18 fits
16:17:09 - [Hyp

/Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/rainfall_acoustic_classification/feature_engineering/plots.py:327: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


16:17:11 - [FeaturePlots] - INFO - Scaled distributions plot saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/IDSM/exp_1_rf/IDSM_E1_rf_scaled_dist.pdf
16:17:11 - [ModelingPlots] - INFO - Generating Dual Confusion Matrix Plot...
16:17:12 - [ModelingPlots] - INFO - Confusion matrix saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/IDSM/exp_1_rf/IDSM_E1_rf_cm.pdf
16:17:12 - [ModelingPlots] - INFO - Generating Multiclass Precision-Recall Curves...
16:17:13 - [ModelingPlots] - INFO - PR-Curve saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/IDSM/exp_1_rf/IDSM_E1_rf_pr_curve.pdf
16:17:13 - [PSD_Baseline_Notebook] - INFO - --- Tuning & Training: LR ---
16:17:13 - [ClassifierFactory] - INFO - CPU Scaling: Requested 1 -> Allocated 1 cores.
16:17:13 - [ClassifierFactory] - INFO - Building registered model architecture: LR
16:17:13 - [HyperparameterTuner] - INFO - Starting Optimizat

...

16:23:44 - [FeaturePlots] - INFO - Scaled distributions plot saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/UECE/exp_5_lr/UECE_E5_lr_scaled_dist.pdf
16:23:44 - [ModelingPlots] - INFO - Generating Dual Confusion Matrix Plot...
16:23:46 - [ModelingPlots] - INFO - Confusion matrix saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/UECE/exp_5_lr/UECE_E5_lr_cm.pdf
16:23:46 - [ModelingPlots] - INFO - Generating Multiclass Precision-Recall Curves...
16:23:46 - [ModelingPlots] - INFO - PR-Curve saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/UECE/exp_5_lr/UECE_E5_lr_pr_curve.pdf
16:23:46 - [PSD_Baseline_Notebook] - INFO - --- Tuning & Training: SGD ---
16:23:46 - [ClassifierFactory] - INFO - CPU Scaling: Requested 1 -> Allocated 1 cores.
16:23:46 - [ClassifierFactory] - INFO - Building registered model architecture: SGD
16:23:46 - [HyperparameterTuner] - INFO - Starting Optimiz

/Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/rainfall_acoustic_classification/feature_engineering/plots.py:327: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


16:23:48 - [FeaturePlots] - INFO - Scaled distributions plot saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/UECE/exp_5_sgd/UECE_E5_sgd_scaled_dist.pdf
16:23:48 - [ModelingPlots] - INFO - Generating Dual Confusion Matrix Plot...
16:23:49 - [ModelingPlots] - INFO - Confusion matrix saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/UECE/exp_5_sgd/UECE_E5_sgd_cm.pdf
16:23:49 - [ModelingPlots] - INFO - Generating Multiclass Precision-Recall Curves...
16:23:50 - [ModelingPlots] - INFO - PR-Curve saved to /Users/fabiomachadomilan/Documents/GitHub/RAC-semiurban-forest-ML-2026/reports/UECE/exp_5_sgd/UECE_E5_sgd_pr_curve.pdf


,Dataset,Exp_ID,Algorithm,Content,Granularity,f1_macro,pr_auc_macro,classification_report_Dry_precision,classification_report_Dry_recall,classification_report_Dry_f1-score,...,classification_report_Moderate_f1-score,classification_report_Moderate_support,classification_report_Violent_precision,classification_report_Violent_recall,classification_report_Violent_f1-score,classification_report_Violent_support,PR_AUC_Class_Heavy,PR_AUC_Class_Light,PR_AUC_Class_Moderate,PR_AUC_Class_Violent
0,IDSM,1,RF,dry/wet,2-Class,0.711239,0.722854,0.904110,0.504202,0.647376,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IDSM,1,LR,dry/wet,2-Class,0.577486,0.720284,0.569089,0.849503,0.681581,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IDSM,1,SGD,dry/wet,2-Class,0.565317,0.720284,0.562312,0.854851,0.678387,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IDSM,2,RF,dry/wet,3-Class,0.616060,0.641136,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IDSM,2,LR,dry/wet,3-Class,0.587003,0.661702,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


16:23:50 - [PSD_Baseline_Notebook] - INFO - Pipeline execution finished. Full metrics, including individual PR-AUCs, are saved.
